# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Мои предсказания и объяснения:

1. 20 и 10 потому что s["b"] берет значение по названию индекса "b" а iloc[0] берет по позиции то есть первый элемент 10
2. Алматы NaN, Астана NaN, Шымкент 3.0 потому что pandas складывает по одинаковым индексам, Шымкент есть в обоих 2+1=3 а Алматы и Астана есть только в одном поэтому NaN, из за NaN тип становится float поэтому 3.0
3. (2, 2) потому что условию x>1 подходят только 2 и 3 значит 2 строки а столбцов как было 2
4. Series DataFrame потому что одна скобка df["x"] возвращает Series а список столбцов df[["x"]] возвращает DataFrame
5. [1, 2] и 5 потому что после фильтра индексы сохраняются и не начинаются заново с 0, iloc[0, 1] берет первую строку фильтра и второй столбец y а там 5

In [32]:
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])

20 10
Алматы     NaN
Астана     NaN
Шымкент    3.0
dtype: float64
(2, 2)
Series DataFrame
[1, 2] 5


In [33]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [34]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [35]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [36]:
# База 1
print(f"Строк: {df.shape[0]}, столбцов: {df.shape[1]}")
print(df.dtypes)
print(f"Пропусков в delivery_days: {df['delivery_days'].isna().sum()}")
print(df[["city", "category", "status", "customer_id"]].nunique())
print(df["city"].value_counts())
print(f"Повторяющихся order_id: {len(df) - df['order_id'].nunique()}")
print(f"Заказов TEST: {(df['customer_id'] == 'TEST').sum()}")
print(f"Самая большая сумма: {df['amount'].max()}")

Строк: 5463, столбцов: 8
order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount           float64
delivery_days    float64
status               str
dtype: object
Пропусков в delivery_days: 160
city             14
category          5
status            2
customer_id    2003
dtype: int64
city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        397
Атырау        146
астана         84
Astana         80
Almaty         69
 Алматы        66
алматы         64
АЛМАТЫ         62
шымкент        54
Shymkent       39
Name: count, dtype: int64
Повторяющихся order_id: 180
Заказов TEST: 3
Самая большая сумма: 9999990.0


Три проблемы:
1. Один город записан по разному (Алматы, алматы, АЛМАТЫ, Almaty...) поэтому городов 14 а не 6, выручка одного города разойдется на несколько строк
2. Есть повторяющиеся order_id, дубли считают одну выручку два раза и завышают отчет
3. Есть TEST заказы с огромной суммой 9 999 990, они раздувают выручку и средний чек

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [37]:
# База 2
ok = df.loc[paid & not_test, "amount"]
print(f"Средний чек: {round(ok.mean())}")
print(f"Медиана: {ok.median()}")
print(f"25-й перцентиль: {ok.quantile(0.25)}")
print(f"75-й перцентиль: {ok.quantile(0.75)}")

Средний чек: 14672
Медиана: 10300.0
25-й перцентиль: 5880.0
75-й перцентиль: 18690.0


Типичный чек это медиана, половина заказов дешевле нее а половина дороже. Среднее выше медианы потому что несколько дорогих заказов тянут его вверх, поэтому в отчете честнее показать медиану

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [38]:
# База 3
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
cancel = df["status"] == "отменён"
sh = df["city"].isin(SH)
al = df["city"].isin(AL)

print(f"Доля отмен всех заказов: {cancel.mean():.2%}")
print(f"Доля отмен в Шымкенте: {cancel[sh].mean():.2%}")
print(f"Доля отмен в Алматы: {cancel[al].mean():.2%}")
print(f"Шымкент выделяется, отмен больше чем в Алматы в {cancel[sh].mean() / cancel[al].mean():.2f} раз")

Доля отмен всех заказов: 10.23%
Доля отмен в Шымкенте: 20.40%
Доля отмен в Алматы: 7.86%
Шымкент выделяется, отмен больше чем в Алматы в 2.59 раз


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [39]:
# Стандарт 4
base = paid & not_test
sh_sep = df.loc[base & sh & sep, "amount"].sum()
sh_oct = df.loc[base & sh & octo, "amount"].sum()
al_sep = df.loc[base & al & sep, "amount"].sum()
al_oct = df.loc[base & al & octo, "amount"].sum()

sh_ch = (sh_oct - sh_sep) / sh_sep
al_ch = (al_oct - al_sep) / al_sep
print(f"Шымкент: сентябрь {sh_sep}, октябрь {sh_oct}, изменение {sh_ch:+.2%}")
print(f"Алматы: сентябрь {al_sep}, октябрь {al_oct}, изменение {al_ch:+.2%}")
print(f"В Шымкенте падение сильнее в {sh_ch / al_ch:.2f} раз, надо проверить что там с доставкой и отменами")

Шымкент: сентябрь 4949800.0, октябрь 4044010.0, изменение -18.30%
Алматы: сентябрь 15523940.0, октябрь 14643020.0, изменение -5.67%
В Шымкенте падение сильнее в 3.22 раз, надо проверить что там с доставкой и отменами


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [40]:
# Стандарт 5
# Ошибка 1 (падает): & выполняется раньше чем ==, поэтому каждое условие надо взять в скобки
# Ошибка 2 (тихая): == "Шымкент" не считает "шымкент" и "Shymkent", поэтому isin(SH)
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print(f"Отмененных заказов в Шымкенте: {len(cancel_sh)}")

Отмененных заказов в Шымкенте: 164


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [41]:
# Стандарт 6
counts = df["customer_id"].value_counts()
print(counts.head(3))
top = counts.index[0]
top_orders = df.loc[df["customer_id"] == top, ["order_id", "order_date", "city", "amount", "status"]]
print(top_orders)
print(f"Повторяющихся order_id у {top}: {len(top_orders) - top_orders['order_id'].nunique()}")

customer_id
C01818    11
C01076     9
C01947     9
Name: count, dtype: int64
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен
Повторяющихся order_id у C01818: 1


Да у самого частого клиента один заказ записан два раза (тот же order_id, время и сумма) это дубль а не новая покупка. Значит вся выручка которую мы считали завышена и дубли надо убрать

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [42]:
# Челлендж 7
q1 = ok.quantile(0.25)
q3 = ok.quantile(0.75)
IQR = q3 - q1
bound = q3 + 1.5 * IQR
out = paid & not_test & (df["amount"] > bound)
print(f"Граница: {bound}")
print(f"Заказов выше границы: {out.sum()}")
print(f"Доля: {out.sum() / len(ok):.2%}")
print(f"Чаще всего категория: {df.loc[out, 'category'].value_counts().index[0]}")

Граница: 37905.0
Заказов выше границы: 320
Доля: 6.53%
Чаще всего категория: Электроника


По IQR выбросов больше чем по z-оценке (87) потому что z-оценка считается через среднее и std а дорогие заказы сами их увеличивают и граница уходит далеко. IQR считается через перцентили и на дорогие заказы почти не реагирует поэтому граница ниже

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [43]:
# Челлендж 8
rate = (df.loc[cancel, "city"].value_counts() / df["city"].value_counts()).sort_values()
print(rate)
print(df["city"].value_counts()[SH])

city
АЛМАТЫ       0.032258
шымкент      0.037037
 Алматы      0.045455
Almaty       0.057971
Атырау       0.075342
Алматы       0.079917
Астана       0.084829
Караганда    0.089219
Astana       0.100000
Актобе       0.103275
астана       0.130952
алматы       0.140625
Шымкент      0.215190
Shymkent     0.230769
Name: count, dtype: float64
city
Шымкент     711
шымкент      54
Shymkent     39
Name: count, dtype: int64


Pandas делит по одинаковым названиям городов. У Шымкента доли у разных написаний разные но у "шымкент" и "Shymkent" очень мало заказов поэтому по одному написанию судить нельзя. До анализа надо привести все написания города к одному виду